In [ ]:
from langgraph.graph import StateGraph , START , END 
from typing import TypedDict , Annotated 
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
from langchain_core.messages import HumanMessage 
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

True

In [4]:
llm = HuggingFaceEndpoint(repo_id ="Qwen/Qwen2.5-7B-Instruct")
model = ChatHuggingFace(llm=llm)

In [5]:
from langgraph.graph.message import add_messages

class Chatstate(TypedDict):
    
    messages : Annotated[list, add_messages]

In [6]:
def chat_node(state:Chatstate):
    
    messages = state['messages']
    
    response = model.invoke(messages)
    
    return {'messages':[response]}

In [7]:
checkpointer = InMemorySaver()

graph = StateGraph(Chatstate)


    
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)


chatbot = graph.compile(checkpointer=checkpointer)


In [ ]:
thread_id = '1'

while True:
    
    user_message = input('Type here: ')
    
    print('user:', user_message)
    
    if user_message.strip().lower() in ['exit','quit','bye','end']:
        break
    
    config = {'configurable' : {'thread_id': thread_id}}
    
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config= config)
    
    print('AI:', response['messages'][-1].content)

user: hii
AI: Hello! How can I assist you today?
user: my name is shubham upadhyay
AI: Hello Shubham, it's nice to meet you! How can I assist you today? Do you have any specific questions or topics you'd like to discuss?
user: whats my full name
AI: Your full name is Shubham Upadhyay. How can I assist you further with this information? Do you have any specific questions or need any help related to your name?
user: exit


: 